# Project 8: Long-Term Memory Agent

Compare recent-window, episodic, and hybrid semantic/temporal memory. Test
corrections, conflicts, evidence, consolidation, and deletion across implemented stores.

Upgrade v1 prevents deleted-event replay, versions answer caches, and separates
retrieval recall from cited-evidence recall. Context size is measured in words.
An optional validated natural-language extractor has event-bound consent and provenance checks.
Model extraction accuracy and provider-level deletion are not established by fixtures.

In [ ]:
from pathlib import Path
import os,subprocess,sys
candidates=[Path.cwd(),Path.cwd()/"project8",Path("/content/ai_agentic_attemptings/project8")]
PROJECT_ROOT=next((p.resolve() for p in candidates if (p/"config/default.json").exists()),None)
if PROJECT_ROOT is None:
    repo=Path("/content/ai_agentic_attemptings")
    if not repo.exists(): subprocess.run(["git","clone","https://github.com/soraber/ai_agentic_attemptings.git",str(repo)],check=True)
    else: subprocess.run(["git","-C",str(repo),"pull","--ff-only"],check=True)
    PROJECT_ROOT=repo/"project8"
os.chdir(PROJECT_ROOT)
if not os.getenv("AI_PROJECT_SKIP_INSTALL"):
    subprocess.run([sys.executable,"-m","pip","install","--upgrade-strategy","only-if-needed","-r","requirements-colab.txt"],check=True)
    subprocess.run([sys.executable,"-m","pip","install","-e",".","--no-deps"],check=True)
source_root=PROJECT_ROOT/"src"
if str(source_root) not in sys.path: sys.path.insert(0,str(source_root))
if not os.getenv("AI_PROJECT_SKIP_INSTALL"):
    check=subprocess.run([sys.executable,"-m","pip","check"],text=True,capture_output=True)
    if check.returncode: print(check.stdout or check.stderr)
from project8_agent.memory import MemoryStore
try: import torch
except ImportError: torch=None
print("Project 8 imports passed")
print({"cuda":bool(torch and torch.cuda.is_available()),"gpu":torch.cuda.get_device_name(0) if torch and torch.cuda.is_available() else None})

In [1]:
import getpass,os,sys
from project8_agent.config import load_config
EVAL_BACKEND="local_gpu"  # deterministic | openai | local_gpu
RUN_FULL_EVAL=True; RUN_API_EVAL=EVAL_BACKEND=="openai"; RUN_LOCAL_GPU_EVAL=EVAL_BACKEND=="local_gpu"; config=load_config(PROJECT_ROOT/"config/default.json")
if RUN_API_EVAL and not os.getenv("OPENAI_API_KEY"):
    if "google.colab" in sys.modules:
        from google.colab import userdata
        key=userdata.get("OPENAI_API_KEY")
    else: key=getpass.getpass("OPENAI_API_KEY (hidden): ")
    if not key: raise RuntimeError("OPENAI_API_KEY required for API mode")
    os.environ["OPENAI_API_KEY"]=key
if RUN_LOCAL_GPU_EVAL:
    import gc,torch
    if not torch.cuda.is_available(): raise RuntimeError("Select a Colab GPU runtime for local_gpu mode")
    stale_names=("source_planner","schema_retriever","local_backend","planners","planner","embedding_retriever","local_answerer")
    released=[name for name in stale_names if globals().pop(name,None) is not None]
    gc.collect(); torch.cuda.empty_cache(); free_bytes,total_bytes=torch.cuda.mem_get_info()
    print({"released_gpu_objects":released,"free_gpu_gib":round(free_bytes/2**30,2),"total_gpu_gib":round(total_bytes/2**30,2)})
print(config.model_dump())

{'released_gpu_objects': [], 'free_gpu_gib': 39.08, 'total_gpu_gib': 39.49}
{'project_id': 'project8', 'seed': 20260802, 'model': 'gpt-5.6-luna', 'reasoning_effort': 'low', 'working_window_size': 6, 'episodic_top_k': 5, 'neighbor_turns': 1, 'qa_per_conversation': 40, 'selected_conversations': 2, 'max_model_calls': 300, 'max_output_tokens': 500, 'max_retries': 2, 'max_estimated_cost_usd': 8.0, 'input_price_per_million_usd': 1.0, 'output_price_per_million_usd': 6.0, 'local_model': 'Qwen/Qwen2.5-7B-Instruct', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'local_device': 'cuda', 'local_max_new_tokens': 300}


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,"tools/fetch_locomo.py"],check=True)
subset_path=PROJECT_ROOT/"data/cache/locomo_subset.json"
print("LoCoMo subset:",subset_path.relative_to(PROJECT_ROOT).as_posix())

In [2]:
import json
subset=json.loads(subset_path.read_text()); lifecycle=json.loads((PROJECT_ROOT/"data/lifecycle_cases.json").read_text())
assert len(subset)==2 and sum(len(item["qa"]) for item in subset)==80
print({"conversations":2,"qa":80,"lifecycle_events":len(lifecycle["events"])})

{'conversations': 2, 'qa': 80, 'lifecycle_events': 7}


In [3]:
from project8_agent.memory import MemoryStore
from project8_agent.schemas import MemoryEvent,MemoryQuery
runtime=PROJECT_ROOT/"output/runtime"; runtime.mkdir(parents=True,exist_ok=True); store=MemoryStore(runtime/"memory.sqlite"); store.reset()
for item in lifecycle["events"]: store.ingest(MemoryEvent.model_validate(item))
query=MemoryQuery.model_validate(lifecycle["queries"][0]); print(store.answer(query,"window").model_dump()); print(store.answer(query,"episodic").model_dump())
embedding_retriever=None
if RUN_LOCAL_GPU_EVAL:
    from project8_agent.local_models import EmbeddingEventRetriever
    from project8_agent.locomo import conversation_events
    embedding_retriever=EmbeddingEventRetriever(config.embedding_model,config.local_device)
    sample_events=conversation_events(subset[0]); print({"dense_retrieval":[item["event_id"] for item in embedding_retriever.retrieve(sample_events,subset[0]["qa"][0]["question"],"hybrid",config.working_window_size,config.episodic_top_k)]})

{'query_id': 'Q01', 'system': 'window', 'answer': None, 'evidence_ids': [], 'abstained': True, 'conflict': False, 'context_words': 12, 'latency_ms': 0.26091200015798677}
{'query_id': 'Q01', 'system': 'episodic', 'answer': 'Boston', 'evidence_ids': ['E01'], 'abstained': False, 'conflict': False, 'context_words': 2, 'latency_ms': 0.41445600072620437}
{'dense_retrieval': ['D9:11', 'D14:34', 'D1:3', 'D5:2', 'D8:22']}


In [4]:
from project8_agent.answer_cache import forget_memory
for event_id in lifecycle["delete_event_ids"]:
    forget_memory(store, event_id, "lifecycle", retrievers=[embedding_retriever] if embedding_retriever else [])
    old = next((item for item in lifecycle["events"] if item["event_id"] == event_id), None)
    if old: assert not store.ingest(MemoryEvent.model_validate(old))
correction=MemoryQuery.model_validate(lifecycle["queries"][1]); conflict=MemoryQuery.model_validate(lifecycle["queries"][2])
print(store.answer(correction,"hybrid").model_dump()); print(store.answer(conflict,"hybrid").model_dump()); assert all(store.deletion_verified(e) for e in lifecycle["delete_event_ids"])

# Authored extraction fixture: validates storage controls, not LLM extraction quality.
from project8_agent.extraction import Extraction,ExtractedFact,RetentionConsent,event_digest,ingest_extraction
natural_store=MemoryStore(runtime/"natural_demo.sqlite")
event=MemoryEvent(event_id="natural-demo-1",session_id="demo",timestamp="2026-01-01T12:00:00+00:00",speaker="demo-user",text="I live in Boston.")
consent=RetentionConsent(subject=event.speaker,event_digest=event_digest(event),allowed_keys=["home_city"],retain=True)
extraction=Extraction(facts=[ExtractedFact(key="home_city",value="Boston",source_event_id=event.event_id,evidence_quote=event.text)])
ingest_extraction(natural_store,event,extraction,consent,"authored-fixture-v1")
forget_memory(natural_store,event.event_id,"natural-demo")
assert natural_store.deletion_verified(event.event_id)
assert not ingest_extraction(natural_store,event,extraction,consent,"authored-fixture-v1")
print("Natural-event consent/provenance/deletion fixture passed; no provider call was made")

{'query_id': 'Q02', 'system': 'hybrid', 'answer': 'green', 'evidence_ids': ['E04'], 'abstained': False, 'conflict': False, 'context_words': 4, 'latency_ms': 0.3899849998560967}
{'query_id': 'Q03', 'system': 'hybrid', 'answer': None, 'evidence_ids': [], 'abstained': True, 'conflict': True, 'context_words': 4, 'latency_ms': 0.5590540004050126}
Natural-event consent/provenance/deletion fixture passed; no provider call was made


In [5]:
import os,subprocess,sys
test_env=os.environ.copy(); test_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"]="1"
try:
    result=subprocess.run([sys.executable,"-m","pytest","-q","tests"],text=True,capture_output=True,env=test_env,timeout=120)
except subprocess.TimeoutExpired as exc:
    raise RuntimeError("Project 8 tests exceeded the 120-second Colab limit") from exc
print(result.stdout)
if result.returncode: print(result.stderr); raise RuntimeError("Project 8 tests failed")


==================================== ERRORS ====================================
_________________ ERROR collecting tests/test_report_values.py _________________
tests/test_report_values.py:6: in <module>
    spec.loader.exec_module(report)
<frozen importlib._bootstrap_external>:936: in exec_module
    ???
<frozen importlib._bootstrap_external>:1073: in get_code
    ???
<frozen importlib._bootstrap_external>:1130: in get_data
    ???
E   FileNotFoundError: [Errno 2] No such file or directory: '/content/portfolio_upgrade_20261117T044102897726Z/agentic/project8/tools/generate_report.py'
=========================== short test summary info ============================
ERROR tests/test_report_values.py - FileNotFoundError: [Errno 2] No such file...
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.42s




In [ ]:
from project8_agent.evaluation import evaluate_lifecycle
from project8_agent.locomo import CachedLocalAnswerer, CachedOpenAIAnswerer, evaluate_locomo
from datetime import datetime, timezone
if not RUN_FULL_EVAL: print("Set RUN_FULL_EVAL=True after P08-C07 passes.")
else:
    result_dir=PROJECT_ROOT/"output/runs/upgrade-v1"/EVAL_BACKEND/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    result_dir.mkdir(parents=True, exist_ok=False)
    lifecycle_summary=evaluate_lifecycle(PROJECT_ROOT/"data/lifecycle_cases.json",runtime/"evaluation.sqlite",runtime/"lifecycle_output")
    if RUN_LOCAL_GPU_EVAL:
        import copy,random
        # Two retrievers, three memory systems: bound worst-case calls before generation.
        qa_limit=min(80,config.max_model_calls//6)
        if qa_limit < len(subset): raise RuntimeError("Model-call budget cannot cover every conversation")
        rng=random.Random(2026); ablation_subset=copy.deepcopy(subset)
        for i,sample in enumerate(ablation_subset):
            count=qa_limit//len(subset)+(i < qa_limit%len(subset))
            sample["qa"]=rng.sample(sample["qa"],min(count,len(sample["qa"])))
        ablation_path=result_dir/"ablation_subset.json"
        ablation_path.write_text(json.dumps(ablation_subset,indent=2))
        local_answerer=CachedLocalAnswerer(config,runtime/"upgrade_v1_local_answer_cache.json")
        summaries={}
        for retrieval_name, retriever in [("lexical", None), ("dense", embedding_retriever)]:
            summaries[retrieval_name]=evaluate_locomo(ablation_path,config,runtime/"unused.json",result_dir/retrieval_name,lifecycle_summary,answerer=local_answerer,retriever=retriever,evaluation_mode="locomo_local_gpu")
        print("Same-answerer retrieval ablation:", summaries)
        result_dir=result_dir/"dense"; summary=summaries["dense"]
    elif RUN_API_EVAL:
        summary=evaluate_locomo(subset_path,config,runtime/"upgrade_v1_api_answer_cache.json",result_dir,lifecycle_summary)
    else:
        summary=evaluate_lifecycle(PROJECT_ROOT/"data/lifecycle_cases.json",runtime/"evaluation.sqlite",result_dir)
    print(summary)

In [ ]:
import json
if "result_dir" not in globals(): raise RuntimeError("Run P08-C08 first; historical results are not reused")
path=result_dir/"project8_representative_samples.json"; print(json.loads(path.read_text()) if path.exists() else "Run P08-C08 first.")

In [ ]:
import subprocess,sys
if "result_dir" not in globals(): raise RuntimeError("Run P08-C08 first; historical results are not reused")
summary_path=result_dir/"project8_final_summary.json"
if summary_path.exists():
    subprocess.run([sys.executable,"tools/generate_report.py","--summary",str(summary_path),"--output",str(result_dir/"project8_report.pdf")],check=True); subprocess.run([sys.executable,"tools/validate_project.py"]+([] if RUN_LOCAL_GPU_EVAL else ["--require-results"]),check=True)
else: print("Measured summary absent; report generation skipped.")